In [ ]:
import polars as pl
from pathlib import Path
import altair as alt

MAKE_PRETRAIN_CSV = False
pretrain_log_file = Path("logs", "pretrain_2026-05-06_09-53-04.log")
pretrain_csv = Path("pretrain_log.csv")

if MAKE_PRETRAIN_CSV:
    with open(pretrain_log_file, "r") as f:
        lines = [line.split("INFO")[1].strip() for line in f if "step" in line]
        data = []
        for line in lines:
            parts = line.split("|")
            step = int(parts[0].split()[1])
            loss = float(parts[1].split()[1])
            acc = float(parts[2].split()[1][:-1])
            episodes = int(parts[3].split()[1])
            time = float(parts[4].split()[0][:-1])
            data.append((step, loss, acc, episodes, time))
        pretrain_df = pl.DataFrame(
            data, schema=["step", "loss", "acc", "episodes", "time"], orient="row"
        )
    pretrain_df.write_csv(pretrain_csv)


MAKE_PPO_CSV = False
ppo_log_file = Path("logs", "ppo_2026-05-06_14-45-50.log")
ppo_csv = Path("ppo_log.csv")

if MAKE_PPO_CSV:
    with open(ppo_log_file, "r") as f:
        lines = [line.split("INFO")[1].strip() for line in f if "entropy" in line]
        data = []
        for line in lines:
            parts = line.split("|")
            it = int(parts[0].split()[1])
            turns = float(parts[1].split()[1])
            policy = float(parts[2].split()[1])
            value = float(parts[3].split()[1])
            entropy = float(parts[4].split()[1])
            time = float(parts[5].split()[0][:-1])
            data.append((it, turns, policy, value, entropy, time))
    ppo_df = pl.DataFrame(
        data,
        schema=["iter", "turns", "policy", "value", "entropy", "time"],
        orient="row",
    )
    ppo_df.write_csv(ppo_csv)

In [97]:
# How big the rolling window should be for the charts
WINDOW_SIZE = 10


def make_chart(
    df: pl.DataFrame,
    y_col: str,
    f,
    window_size: int = WINDOW_SIZE,
    arg_col: str = "step",
    scale_x: bool = True,
) -> alt.LayerChart:
    df = df.with_columns(
        pl.col(y_col).rolling_mean(window_size=window_size).alias(f"{y_col}_rolling"),
    )
    # Chart size calculations
    x_max = df[arg_col].max() * 1.05
    x_axis = alt.Axis()
    if scale_x:
        x_axis = alt.Axis(labelExpr="datum.value / 1000 + 'k'")
    x_scale = alt.Scale(domainMax=x_max)

    # Define color scale for lines and rules
    color_domain = ["Rolling avg", "Best rolling", "Best raw"]
    color_range = ["steelblue", "grey", "red"]
    color_scale = alt.Scale(domain=color_domain, range=color_range)
    # Rolling average line
    line = (
        alt.Chart(df.with_columns(pl.lit("Rolling avg").alias("series")))
        .mark_line()
        .encode(
            x=alt.X(arg_col, title=arg_col.capitalize(), axis=x_axis, scale=x_scale),
            y=alt.Y(
                f"{y_col}_rolling",
                title=f"{y_col.capitalize()} ({window_size}-{arg_col} rolling avg)",
            ),
            color=alt.Color("series:N", scale=color_scale, legend=alt.Legend(title="")),
        )
        .properties(
            title=f"{y_col.capitalize()} v {arg_col.capitalize()}",
            width=500,
            height=300,
        )
    )

    # Best rolling and raw value rules
    h_rolling_val = df.select(f(f"{y_col}_rolling")).item()
    rule_rolling = (
        alt.Chart(pl.DataFrame({"y": [h_rolling_val], "series": ["Best rolling"]}))
        .mark_rule(fillOpacity=0.8, strokeDash=[4, 4])
        .encode(
            y=alt.Y("y:Q"),
            color=alt.Color("series:N", scale=color_scale, legend=alt.Legend(title="")),
        )
    )
    h_val = df.select(f(y_col)).item()
    rule = (
        alt.Chart(pl.DataFrame({"y": [h_val], "series": ["Best raw"]}))
        .mark_rule(fillOpacity=0.8, strokeDash=[4, 4])
        .encode(
            y=alt.Y("y:Q"),
            color=alt.Color("series:N", scale=color_scale, legend=alt.Legend(title="")),
        )
    )

    return line + rule_rolling + rule

---

# Imitation Pretraining Analysis


In [98]:
pretrain_df = pl.read_csv(pretrain_csv)
pretrain_df.show()

step,loss,acc,episodes,time
i64,f64,f64,i64,f64
1,4.6508,0.0,0,0.1
1000,1.9606,51.6,21,15.6
2000,1.3671,60.6,41,15.6
3000,1.476,62.5,63,15.5
4000,1.6115,63.5,83,15.1


In [99]:
loss = make_chart(pretrain_df, "loss", pl.min)
acc = make_chart(pretrain_df, "acc", pl.max)

# Total time
total_seconds = pretrain_df["time"].sum()
h, m = divmod(total_seconds, 3600)
m, s = divmod(m, 60)
print(f"Total training time: {int(h)}h {int(m)}m {int(s)}s")
# Min loss and max accuracy
min_loss = pretrain_df["loss"].min()
max_acc = pretrain_df["acc"].max()
print(f"Min loss: {min_loss:.4f} | Max accuracy: {max_acc:.1f}%")
# Display charts
title = alt.TitleParams(
    "Transformer PPO - Imitation Pretraining Metrics", anchor="middle"
)
(loss | acc).resolve_scale(y="independent").properties(title=title)

Total training time: 2h 15m 28s
Min loss: 0.3123 | Max accuracy: 93.1%


alt.HConcatChart(...)

---

# PPO Analysis


In [100]:
ppo_df = pl.read_csv(ppo_csv)
ppo_df.show()

iter,turns,policy,value,entropy,time
i64,f64,f64,f64,f64,f64
1,47.1,0.5675,0.6084,0.0455,48.38
2,47.3,0.0385,0.6059,0.0363,45.72
3,44.0,0.02,0.5883,0.0567,43.33
4,46.9,0.0275,0.563,0.0462,45.12
5,45.8,0.0351,0.5448,0.0364,45.33


In [101]:
turns, policy, value, entropy = [
    make_chart(ppo_df, col, f, arg_col="iter", scale_x=False, window_size=5)
    for col, f in [
        ("turns", pl.min),
        ("policy", pl.min),
        ("value", pl.max),
        ("entropy", pl.max),
    ]
]

# Total time
total_seconds = ppo_df["time"].sum()
h, m = divmod(total_seconds, 3600)
m, s = divmod(m, 60)
print(f"Total training time: {int(h)}h {int(m)}m {int(s)}s")
# Display charts
title = alt.TitleParams("Transformer PPO - Training Metrics", anchor="middle")
((turns | value) & (policy | entropy)).resolve_scale(y="independent").properties(
    title=title
)

Total training time: 1h 15m 7s


alt.VConcatChart(...)